In [7]:
import pandas as pd
import re

pd.set_option('display.max_colwidth', 80)


STORIES_PATH   = r"C:\Users\East-Sound\Downloads\hacker-news-airflow\data-lake\processed\stories\part-00000-4e961818-a82a-48ad-adeb-f2de76adde9f-c000.snappy.parquet"
COMMENTS1_PATH = r"C:\Users\East-Sound\Downloads\hacker-news-airflow\data-lake\processed\comments\part-00000-90f9b878-79c8-4079-a87b-d138727bec8b-c000.snappy.parquet"
COMMENTS2_PATH = r"C:\Users\East-Sound\Downloads\hacker-news-airflow\data-lake\processed\comments\part-00001-90f9b878-79c8-4079-a87b-d138727bec8b-c000.snappy.parquet"

In [8]:
stories = pd.read_parquet(STORIES_PATH)
comments = pd.concat(
    [pd.read_parquet(COMMENTS1_PATH), pd.read_parquet(COMMENTS2_PATH)],
    ignore_index=True
)

print("Stories:", stories.shape)
print("Comments:", comments.shape)
stories.head()


Stories: (99, 8)
Comments: (2324, 5)


,id,title,url,score,by,time,descendants,kids
0,49488224,How bicycle coaster brakes work (2018),https://www.dougbarnesauthor.com/2018/06/how-bicycle-coaster-brakes-work.html,89,Vedor,2026-08-29 09:05:53,87,"[49527465, 49527351, 49531602, 49527449, 49526975, 49533622, 49535072, 49527..."
1,49489098,Building Autonomous Goal Loops That Deliver,https://jx0.ca/building-autonomous-goal-loops-that-deliver/,53,jarredkenny,2026-08-29 11:49:32,9,"[49531359, 49540863, 49532590, 49532099, 49532427, 49531498, 49531431]"
2,49490498,Magic eye tube,https://en.wikipedia.org/wiki/Magic_eye_tube,101,peter_d_sherman,2026-08-29 15:03:41,27,"[49525992, 49553340, 49528101, 49527070, 49526424, 49528726, 49527299, 49527..."
3,49493468,American Airlines mechanic Azriel “Al” Blackman has died,https://simpleflying.com/american-airlines-mechanic-passes-away-100-record-8...,383,NaOH,2026-08-29 21:25:17,163,"[49527756, 49521522, 49493475, 49521729, 49519762, 49520351, 49526918, 49524..."
4,49494236,Open Battery Information,https://github.com/mnh-jansson/open-battery-information,56,toomuchtodo,2026-08-29 23:27:41,11,"[49534651, 49535589, 49534629, 49494237, 49534722, 49534376, 49534511, 49534..."


In [9]:
comments.head()

,id,story_id,by,text,time
0,49472781,49516059,Rendello,"This one was good:<p><i>Interview: Doug McIlroy</i>: <a href=""https:&#x2F;&#...",2026-08-28 00:00:40
1,49473822,49516059,vivzkestrel,- i like visual<p>- can we get a whole magazine that explains ideas like thi...,2026-08-28 02:55:04
2,49475389,49516059,bargainbin,Love the aesthetic. There was another ezine like this years ago focused on c...,2026-08-28 07:17:29
3,49490508,49490498,peter_d_sherman,"Related:<p>The Magic-Eye Tuning Indicator, W G Morley, The Radio Constructor...",2026-08-29 15:05:05
4,49493475,49493468,toomuchtodo,"Azriel &quot;Al&quot; Blackman<p><a href=""https:&#x2F;&#x2F;www.nytimes.com&...",2026-08-29 21:26:30


In [11]:
import urllib.parse

# 1. استخراج الـ domain من رابط الـ url (إذا كان موجود)
def extract_domain(url):
    if pd.isna(url) or not url:
        return None
    try:
        netloc = urllib.parse.urlparse(str(url)).netloc
        return netloc.replace('www.', '')
    except Exception:
        return None

# إضافة عمود الـ domain
stories['domain'] = stories['url'].apply(extract_domain)

# 2. تغيير اسم عمود التعليقات من descendants إلى comment_count (لو موجود)
if 'descendants' in stories.columns:
    stories['comment_count'] = stories['descendants']
else:
    stories['comment_count'] = 0

# 3. عرض أعلى 15 خبر
top_stories = (
    stories.sort_values('score', ascending=False)
    [['id', 'title', 'domain', 'by', 'score', 'comment_count', 'time']]
    .reset_index(drop=True)
)

top_stories.head(15)

,id,title,domain,by,score,comment_count,time
0,49503601,“I just chose words carefully”,unsung.aresluna.org,zdw,1252,353,2026-08-30 22:49:48
1,49520022,AnkiDroid: Google Play no longer allowing Open Collective donation link,github.com,hexa555,917,278,2026-09-01 10:11:02
2,49526069,How accurate have Ed Zitron's AI skeptic predictions been?,danluu.com,jatins,864,1034,2026-09-01 18:35:15
3,49517448,Fastpotify,fastpotify.rocks,nreece,847,560,2026-09-01 02:52:14
4,49535752,A note on subscription prices from LWN,lwn.net,rwky,721,148,2026-09-02 13:17:12
5,49519939,I trained a small transformer in 1.5hrs and it beats many LLMs,mvakde.github.io,porridgeraisin,662,165,2026-09-01 09:52:45
6,49521973,Introducing Ad Blocker for Firefox on iOS,blog.mozilla.org,HieronymusBosch,581,204,2026-09-01 13:46:49
7,49523754,"Play Store blocks AuroraStore, hurting GrapheneOS users",gitlab.com,erikvanoosten,560,263,2026-09-01 15:55:53
8,49529621,FBI Probes Service Selling 153M+ Drivers Licenses,krebsonsecurity.com,tatersolid,412,274,2026-09-01 23:17:43
9,49506819,Breaking Claude Code Opus 5 Auto Mode,embracethered.com,Recursing,397,121,2026-08-31 07:49:18


In [12]:
top_stories = (
    stories.sort_values('score', ascending=False)
    [['id', 'title', 'domain', 'by', 'score', 'comment_count', 'time']]
    .reset_index(drop=True)
)
top_stories.head(15)


,id,title,domain,by,score,comment_count,time
0,49503601,“I just chose words carefully”,unsung.aresluna.org,zdw,1252,353,2026-08-30 22:49:48
1,49520022,AnkiDroid: Google Play no longer allowing Open Collective donation link,github.com,hexa555,917,278,2026-09-01 10:11:02
2,49526069,How accurate have Ed Zitron's AI skeptic predictions been?,danluu.com,jatins,864,1034,2026-09-01 18:35:15
3,49517448,Fastpotify,fastpotify.rocks,nreece,847,560,2026-09-01 02:52:14
4,49535752,A note on subscription prices from LWN,lwn.net,rwky,721,148,2026-09-02 13:17:12
5,49519939,I trained a small transformer in 1.5hrs and it beats many LLMs,mvakde.github.io,porridgeraisin,662,165,2026-09-01 09:52:45
6,49521973,Introducing Ad Blocker for Firefox on iOS,blog.mozilla.org,HieronymusBosch,581,204,2026-09-01 13:46:49
7,49523754,"Play Store blocks AuroraStore, hurting GrapheneOS users",gitlab.com,erikvanoosten,560,263,2026-09-01 15:55:53
8,49529621,FBI Probes Service Selling 153M+ Drivers Licenses,krebsonsecurity.com,tatersolid,412,274,2026-09-01 23:17:43
9,49506819,Breaking Claude Code Opus 5 Auto Mode,embracethered.com,Recursing,397,121,2026-08-31 07:49:18


In [13]:
top_authors = (
    stories.groupby('by')
    .agg(
        stories_posted=('id', 'count'),
        total_score=('score', 'sum'),
        avg_score=('score', 'mean'),
        total_comments=('comment_count', 'sum'),
    )
    .sort_values('stories_posted', ascending=False)
    .reset_index()
)
top_authors.head(15)


,by,stories_posted,total_score,avg_score,total_comments
0,speckx,3,55,18.333333,15
1,porridgeraisin,3,802,267.333333,209
2,Brajeshwar,2,48,24.000000,20
3,ilamont,2,208,104.000000,150
4,duck,2,97,48.500000,17
5,zdw,2,1381,690.500000,421
6,Bender,1,29,29.000000,6
7,Anon84,1,151,151.000000,174
8,AntonioLi,1,199,199.000000,66
9,Alephinitesimal,1,257,257.000000,251


In [14]:
top_commenters = (
    comments.groupby('by')
    .size()
    .rename('comments_posted')
    .sort_values(ascending=False)
    .reset_index()
)
top_commenters.head(15)


,by,comments_posted
0,thataccount,8
1,andai,6
2,ChrisArchitect,6
3,VCFundedGenYer,6
4,josefritzishere,5
5,colincowardly,4
6,dvt,4
7,dofm,4
8,jdw64,4
9,luciana1u,4


In [15]:
top_domains = (
    stories['domain']
    .value_counts()
    .rename_axis('domain')
    .reset_index(name='story_count')
)
top_domains.head(15)


,domain,story_count
0,github.com,4
1,nytimes.com,3
2,twitter.com,2
3,bbc.com,2
4,dougbarnesauthor.com,1
5,jx0.ca,1
6,en.wikipedia.org,1
7,simpleflying.com,1
8,chosun.com,1
9,nature.com,1


In [17]:
# 1. تحويل عمود time لتاريخ وإنشاء عمود date
stories['date'] = pd.to_datetime(stories['time']).dt.date

# 2. التجميع حسب التاريخ
stories_per_day = (
    stories.groupby('date')
    .size()
    .rename('story_count')
    .reset_index()
)

stories_per_day

,date,story_count
0,2026-08-29,5
1,2026-08-30,4
2,2026-08-31,9
3,2026-09-01,21
4,2026-09-02,30
5,2026-09-03,30


In [19]:
# 1. استخراج الساعة من عمود time
stories['hour'] = pd.to_datetime(stories['time']).dt.hour

# 2. حساب المتوسط وعدد الأخبار لكل ساعة
score_by_hour = (
    stories.groupby('hour')
    .agg(
        avg_score=('score', 'mean'),
        stories_count=('id', 'count')
    )
    .reset_index()
    .sort_values('hour')
)

score_by_hour

,hour,avg_score,stories_count
0,1,128.000000,3
1,2,338.750000,4
2,3,109.000000,1
3,4,290.000000,1
4,5,191.000000,1
5,6,283.000000,1
6,7,397.000000,1
7,8,350.000000,1
8,9,205.750000,4
9,10,917.000000,1


In [21]:
# 1. حساب طول نص التعليق وإنشاء عمود comment_length
comments['comment_length'] = comments['text'].str.len().fillna(0)

# 2. ترتيب التعليقات وعرض أطول 15 تعليق
longest_comments = (
    comments.sort_values('comment_length', ascending=False)
    [['by', 'comment_length', 'time', 'text']]  # اخترنا الأعمدة الموجودة فعلياً
    .head(15)
    .reset_index(drop=True)
)

longest_comments

,by,comment_length,time,text
0,prman_,7743,2026-09-01 16:05:42,Location: Brazil (UTC-3)<p>Remote: Yes. Happy to overlap substantially with ...
1,ineedasername,4086,2026-09-01 15:39:46,Location: Remote&#x2F;Hybrid NYC area<p>Resume&#x2F;CV&#x2F;Experience: work...
2,dabluecaboose,3949,2026-09-02 16:24:36,Rocket Scientist here. Quick (simplified) primer on engine nozzles and why ...
3,iLemming,3551,2026-08-31 02:58:58,"Almost irrelevant to the story itself, but here&#x27;s what it feels like to..."
4,_shantaram,3474,2026-09-02 19:09:47,"Location: Bangalore, India<p>Remote: Yes. I have worked remote-only for six ..."
5,varun636,3434,2026-09-01 15:23:20,"Location: Vijayawada, India<p>Remote: Yes (2pm-11pm IST — full EU hours + US..."
6,nater5000,3364,2026-09-03 15:58:48,Just terrible.<p>The author seems to conflate not caring about the inconsequ...
7,squidlib,3303,2026-09-03 04:27:20,"This is an ad, so it&#x27;s going to be fluff, but the example use cases rea..."
8,xtracto,3007,2026-09-01 22:15:24,"<p><pre><code> Location: Mexico, UTC-6\n Remote: Yes. \n Willing t..."
9,entaroadun123,2902,2026-09-02 18:05:42,"Location: Renton, Washington (US Pacific)<p>Remote: Yes, remote only<p>Willi..."


In [22]:
summary = pd.DataFrame({
    'Metric': [
        'Total Stories', 'Total Comments', 'Unique Story Authors', 'Unique Commenters',
        'Avg Score per Story', 'Max Score', 'Avg Comments per Story',
        'Date Range Start', 'Date Range End'
    ],
    'Value': [
        len(stories), len(comments), stories['by'].nunique(), comments['by'].nunique(),
        round(stories['score'].mean(), 1), stories['score'].max(),
        round(stories['comment_count'].mean(), 1),
        str(stories['time'].min()), str(stories['time'].max())
    ]
})
summary


,Metric,Value
0,Total Stories,99
1,Total Comments,2324
2,Unique Story Authors,91
3,Unique Commenters,2078
4,Avg Score per Story,174.0
5,Max Score,1252
6,Avg Comments per Story,89.5
7,Date Range Start,2026-08-29 09:05:53
8,Date Range End,2026-09-03 22:20:39


In [24]:
!pip install openpyxl

     ------------------------------------ 250.9/250.9 kB 904.8 kB/s eta 0:00:00



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import os
os.makedirs('powerbi_export', exist_ok=True)

stories.to_csv('powerbi_export/stories.csv', index=False)
comments.to_csv('powerbi_export/comments.csv', index=False)
top_stories.to_csv('powerbi_export/top_stories.csv', index=False)
top_authors.to_csv('powerbi_export/top_authors.csv', index=False)
top_commenters.to_csv('powerbi_export/top_commenters.csv', index=False)
top_domains.to_csv('powerbi_export/top_domains.csv', index=False)
stories_per_day.to_csv('powerbi_export/stories_per_day.csv', index=False)
score_by_hour.to_csv('powerbi_export/score_by_hour.csv', index=False)
longest_comments.to_csv('powerbi_export/longest_comments.csv', index=False)
summary.to_csv('powerbi_export/summary.csv', index=False)

with pd.ExcelWriter('powerbi_export/hackernews_analysis.xlsx', engine='openpyxl') as writer:
    summary.to_excel(writer, sheet_name='Summary', index=False)
    stories.to_excel(writer, sheet_name='Stories', index=False)
    comments.to_excel(writer, sheet_name='Comments', index=False)
    top_stories.to_excel(writer, sheet_name='Top Stories', index=False)
    top_authors.to_excel(writer, sheet_name='Top Authors', index=False)
    top_commenters.to_excel(writer, sheet_name='Top Commenters', index=False)
    top_domains.to_excel(writer, sheet_name='Top Domains', index=False)
    stories_per_day.to_excel(writer, sheet_name='Stories Per Day', index=False)
    score_by_hour.to_excel(writer, sheet_name='Score By Hour', index=False)

print("Export done. Files are in the 'powerbi_export' folder:")
for f in sorted(os.listdir('powerbi_export')):
    print(' -', f)


Export done. Files are in the 'powerbi_export' folder:
 - comments.csv
 - hackernews_analysis.xlsx
 - longest_comments.csv
 - score_by_hour.csv
 - stories.csv
 - stories_per_day.csv
 - summary.csv
 - top_authors.csv
 - top_commenters.csv
 - top_domains.csv
 - top_stories.csv
